# Tarea 1

## Objetivo:

Implementar y contrastar dos modelos de clasificación —uno explicativo y otro predictivo— bajo criterios de rigor técnico y buenas prácticas de Data Mining y Modelización Predictiva.

### Sub-objetivos:
- Diferenciar y contrastar el enfoque de un modelo explicativo frente a uno predictivo en problemas de clasificación.
- Identificar y extraer los patrones clave de las features que determinan el comportamiento de la variable target.
- Optimizar el rendimiento predictivo mediante transformaciones de las features.
- Desarrollar un código limpio y estructurado, aplicando un manejo de errores eficiente para asegurar que el pipeline de datos no se rompa.
- Utilizar de forma autónoma la documentación de _scikit-learn_ y _statsmodels_ para implementar los algoritmos y resolver dudas técnicas.

## Tarea del estudiante:

Lee detenidamente las siguientes instrucciones.

El estudiante tendrá bastante libertad para realizar la tarea, utilizando sus propios criterios y estrategias. No obstante,

- deberá estar limitado a las librerías de _pandas_, _numpy_, _scikit-learn_ y _statsmodels_;
- no modificará los comentarios de los enunciados, pero sí podrá editar y crear celdas para completar cada una de las partes;
- en los apartados que se solicita "explicación" o "comentario", no olvidar dejar vuestra reflexión, si no estuviese, no se puntuaría ese apartado.

Una vez finalizada la actividad, guarda tu fichero, reinicia el kernel vuélvelo a ejecutar. No deben aparecer errores. Un notebook con errores es una Tarea incompleta. Para la entrega, TODO EL CÓDIGO DEBE ESTAR EJECUTADO (EN ORDEN) EN EL FICHERO QUE ENTREGÁIS.

RECUERDA SUBIR CADA UNO DE LOS FICHEROS .ipynb TAL CUAL (sueltos), SIN COMPRIMIR Y SIN CAMBIARLES EL NOMBRE. Los ficheros subidos deben tener exactamente el mismo nombre de fichero que tenían cuando los recibiste. No subas ningún PDF ni ningún fichero ZIP ni nada similar. La plataforma ya los separa automáticamente en carpetas que traen el nombre y apellidos del alumno, por lo que NO es necesario que lo pongas en ninguna parte.

## Evaluación:

- Este notebook se evalúa sobre 10 puntos.

# Parte 0: Librerías

In [10]:
# Se reserva este espacio para la importación de librerías

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

seed = 99

# el estudiante puede añadir aquí las librerías que necesite

# Parte 0: Dataset

Los dos trabajos solitados se harán con el siguiente dataset de "Cancelación de reservas de hotel". Cada registro supone una reserva que ha sido cancelada o disfrutada por parte de clientes. A continuación, se da más detalle sobre las columnas:

**target**:
- *reserva_cancelada*: Indica si la reserva fue cancelada (1) o no cancelada (0).

**features**:
- *dias_antelacion*: Número de días que transcurren entre la fecha en la que el cliente hace la reserva y la fecha de su llegada al hotel.
- *fecha_llegada*: Fecha de comienzo de la reserva.
- *noches_estancia*: Noches totales de la reserva.
- *estado_reserva*: Estado final de la reserva. Valores: Canceled (el cliente canceló), Check-Out (el cliente disfrutó la reserva), No-Show (el cliente nunca apareció ni avisó).
- *fecha_estado_reserva*: Fecha de registro de la variable estado_reserva.
- *adultos*: Número de adultos incluidos en la reserva.
- *ninos*: Número de niños.
- *bebes*: Número de bebés.
- *regimen_alimenticio*: Régimen alimenticio contratado (BB: Solo desayuno, HB: Media pensión, FB: Pensión completa, SC/Undefined: Sin comidas).
- *tipo_habitacion_reservada*: Tipo de habitación que el cliente solicitó originalmente (codificado por letras: A, B, C...).
- *tipo_habitacion_asignada*: Tipo de habitación que el hotel le asignó finalmente en el momento del check-in.
- *plazas_parking_requeridas*: Número de plazas de parking que solicita el cliente.
- *peticiones_especiales_totales*: Número de peticiones especiales realizadas (ej. cuna, cama extra, piso alto).
- *pais_origen*: País de origen del cliente (en formato de código internacional de tres letras, ej. PRT, ESP, FRA).
- *segmento_mercado*: Segmento de mercado al que pertenece la reserva (ej. Directo, Grupos, Agencias de viaje corporativas, etc.).
- *canal_distribucion*: Canal por el que llegó la reserva al hotel (ej. TA/TO: Agencias/Operadores, Directo, GDS, etc.).
- *cliente_repetido*: Variable binaria (0 o 1). Indica si el cliente ya se ha alojado en este hotel en el pasado.
- *cancelaciones_previas*: Número de reservas que este cliente ha cancelado en el pasado antes de la reserva actual.
- *reservas_previas_no_canceladas*: Número de reservas anteriores que el cliente realizó y sí completó con éxito (no canceló).
- *tipo_deposito*: Indica si el cliente realizó un depósito para asegurar la reserva (No Deposit, Non Refundable [no reembolsable], Refundable [reembolsable]).
- *cambios_reserva*: Número de modificaciones o cambios hechos a la reserva desde que se creó hasta el día de la llegada.
- *dias_lista_espera*: Número de días que la reserva estuvo retenida en lista de espera antes de ser confirmada.
- *tipo_cliente*: Tipo de cliente/reserva (Transient: Particular suelto, Group: Grupo, Contract: Tarifa corporativa recurrente, Transient-Party: Particular ligado a grupo).
- *precio_medio_diario*: Precio medio diario (Average Daily Rate) que paga el cliente por noche de habitación.

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/refs/heads/main/reservas_hotel.csv")
df.head()

,reserva_cancelada,dias_antelacion,adultos,ninos,bebes,regimen_alimenticio,pais_origen,segmento_mercado,canal_distribucion,cliente_repetido,cancelaciones_previas,reservas_previas_no_canceladas,tipo_habitacion_reservada,tipo_habitacion_asignada,cambios_reserva,tipo_deposito,dias_lista_espera,tipo_cliente,precio_medio_diario,plazas_parking_requeridas,peticiones_especiales_totales,estado_reserva,fecha_estado_reserva,fecha_llegada,noches_estancia
0,0,342,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-06,0
1,0,737,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-06,0
2,0,7,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-06,1
3,0,13,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-06,1
4,0,14,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-06,2


In [6]:
df.shape

(119390, 25)

In [8]:
# Todo el trabajo del estudiante se realizará con df1.
# Mantendremos la variable df para comparar cuando sea necesario.

df1 = df.copy()

# Parte 1: Tratamiento previo del dataset (EDA + Feature Engineering)

## Variables de fecha (2 puntos)
En este apartado se pide identificar las variables de fecha y crear nuevas features que puedan ser de interés para encontrar "patrones" de cancelación.

Para ello, se pide trabajar **exclusivamente** las columnas de fecha. Al final de cada celda, se eliminará la columna de fecha y se pondrá un comentario por cada una de las **nuevas features** que se hayan creado explicando qué significa. Tenéis total libertad para abordarlo según vuestro propio criterio y considerar la estrategia que mejor convenga.

<div class="alert alert-block alert-danger">
<b>Importante:</b> Máxima atención a "data leakage", utilizar información del "futuro" o datos a los que no tendría acceso en el mundo real para construir modelos predictivos o explicativos puede invalidar totalmente cualquier trabajo. En concreto, si usamos data leakage, esta tarea se puntuará con 0.0. 
</div>

In [ ]:
# Código del estudiante



# Eliminación de las columnas de fecha
df1.drop(columns=..., inplace=True)

In [ ]:
# Explicación de las nuevas columnas creadas
...

## Data split (0 puntos)

En este apartado se pide separar el dataset original en train/test, dando un 20% de los datos al conjunto de test (recordad usar semilla).

<div class="alert alert-block alert-danger">
<b>Importante:</b> Máxima atención a "data leakage", utilizar información del conjunto de test para ajustar el modelo o realizar ajustes con el conjunto de test (los _.fit()_) supone un error grave a la hora de analizar los datos y ajustar los modelos. En concreto, si usamos data leakage, esta tarea se puntuará con 0.0. 
</div>

In [ ]:
# Código del estudiante

X_train, X_test, y_train, y_test = ...

## Imputación de nulos (0 puntos)

No podremos avanzar demasiado si seguimos teniendo nulos. Por ello, se pide detectar los nulos en el dataset y tomar una decisión sobre las actuaciones que llevéis a cabo. Tenéis total libertad para abordarlo según vuestro propio criterio y considerar la estrategia que mejor convenga.

In [ ]:
# Código del estudiante



In [ ]:
# Explicación de las imputaciones realizadas
...

## Exploración de features y creación de nuevas features (1 punto)

En este apartado se pide explorar con plots todas las features

In [9]:
# Código del estudiante (opcional)

cat_cols = ...
ord_cols = ...
num_cols = ...

In [13]:
# Código del estudiante



A continuación, si consideráis que hay que transformar features, puede hacerlo en este apartado. Tenéis total libertad para abordarlo según vuestro propio criterio y considerar la estrategia que mejor convenga (OneHotEncoder, LabelEncoder, PowerTransformer, KBinDiscretizer, etc). Tenéis total libertad para abordarlo según vuestro propio criterio y considerar la estrategia que mejor convenga, incluida cualquier otra funcion de _scikit-learn_ o cualquier transformación customizada acorde a vuestro criterio.

<div class="alert alert-block alert-warning">
<b>Importante:</b> Cuidado con las transformaciones, puede que no interese hacer las mismas transformaciones para un modelo "explicativo" que para uno "predictivo". En este apartado, se recomienda hacer <b>solo</b> las transformaciones que sean útiles para ambos tipos de trabajo. 
</div>

In [ ]:
# Código del estudiante



In [ ]:
# Explicación de las nuevas features o las transformaciones realizadas
...

Finalmente, se os pedirá realizar otra vez los plots de todas las features (se asume que se han modificado algunas, en caso contrario no sería necesario trabajar la siguiente celda).

In [ ]:
# Código del estudiante (opcional)

cat_cols = ...
ord_cols = ...
num_cols = ...

In [ ]:
# Código del estudiante



# Parte 2: Modelo Explicativo (3 puntos)

Dado que la muestra es tan grande. Podremos realizar perfectamente un modelo explicativo con el conjunto de train. No usaremos, por contra, el conjunto de test.

Así, se pedirá en este apartado realizar todo el análisis necesario exclusivamente con la librería de _statsmodels_ para ajustar un modelo explicativo de regresión logística. Para ello, es posible que tengáis que eliminar algunas features, modificar alguna existente, crear alguna nueva, trabajar outliers, plots, análisis, etc. 

Tenéis total libertad para elaborar el modelo explicativo con el conjunto de train según vuestro propio criterio. Todas las iteraciones y decisiones que toméis, estarán aquí reflejadas y comentadas por vuestra parte. Podéis crear todas las celdas que consideréis necesarias. 

In [ ]:
# Código del estudiante



In [ ]:
# Código del estudiante



In [ ]:
# Código del estudiante



In [ ]:
# Código del estudiante



Modelo de referencia. En la última de vuestras celdas dejad claro cuál es el modelo final de referencia a continuación.

In [ ]:
# Código del estudiante 




Para finalizar este apartado, tendréis que aportar un resumen de las conclusiones de vuestro estudio. Se permite un máximo de 5 bullets.

In [14]:
# Comentarios del estudiante
# - Conclusión 1
# - Conclusión 2
# - Conclusión 3
# - Conclusión 4
# - Conclusión 5

# Parte 3: Modelo Predictivo (4 puntos)

Aquí realizaremos un modelo predictivo que entrenaremos con el conjunto de train. Así, se pedirá en este apartado realizar todo el trabajo necesario exclusivamente con la librería de _sklearn_ para ajustar un modelo predictivo de regresión logística. Para ello, es posible que tengáis que eliminar algunas features, modificar alguna existente, crear alguna nueva, trabajar outliers, plots, análisis, etc.

In [15]:
# Código del estudiante



In [ ]:
# Código del estudiante



In [ ]:
# Código del estudiante



Finalmente, se pedirá utilizar el conjunto de test para evaluar la métrica del modelo. Se medirá el accuracy **exclusivamente sobre el conjunto de test**. Pero tenéis total libertad para estudiar si se puede mejorar el umbral de probabilidad de la regresión lineal. 

In [ ]:
# Código del estudiante



In [ ]:
# Código del estudiante

acc = ...
print(f"Final accuracy={acc:.3f}")